In [1]:
import requests

r = requests.get(
    "https://en.wikipedia.org/api/rest_v1/page/summary/Animal",
    headers={"User-Agent": "MyOntologyProject/1.0"}
)
r.json()["extract"]

'Animals are multicellular, eukaryotic organisms belonging to the biological kingdom Animalia. With few exceptions, animals consume organic material, breathe oxygen, have myocytes and are able to move, can reproduce sexually, and grow from a hollow sphere of cells, the blastula, during embryonic development. Animals form a clade, meaning that they arose from a single common ancestor. Over 1.5 million living animal species have been described, of which around 1.05 million are insects, over 85,000 are molluscs, and around 65,000 are vertebrates. It has been estimated there are as many as 7.77\xa0million animal species on Earth. Animal body lengths range from 8.5\xa0μm (0.00033\xa0in) to 33.6\xa0m (110\xa0ft). They have complex ecologies and interactions with each other and their environments, forming intricate food webs. The scientific study of animals is known as zoology, and the study of animal behaviour is known as ethology.'

In [12]:


from owlready2 import get_ontology, Restriction, Or, ThingClass
from dotenv import load_dotenv
import os
import mlflow
import logfire
from pydantic import BaseModel
from pydantic_ai import Agent
import requests
from OntologyCreation import DATA_PATH
import json

In [1]:
import requests, re

headers = {"User-Agent": "OntologyCreation/1.0"}
r = requests.get("https://en.wikipedia.org/api/rest_v1/page/summary/tabby", headers=headers)
print("status:", r.status_code)
extract = r.json().get("extract", "")
print("has may refer to:", "may refer to" in extract.lower())
print("extract:", extract[:200])

status: 200
has may refer to: False
extract: A tabby cat, or simply tabby, describes any domestic cat with a coat pattern distinguished by an M-shaped marking on its forehead, stripes by its eyes, cheeks, along its back, and around its legs and 


In [18]:

query = """
SELECT DISTINCT ?subclass ?parent WHERE {
    ?subclass rdfs:subClassOf ?parent .
    ?parent rdfs:subClassOf* <http://dbpedia.org/ontology/CONCEPT> .
}
LIMIT 30
""".replace("CONCEPT", start_concept)

r = requests.get(
    "https://dbpedia.org/sparql",
    params={"query": query, "format": "json"}
)

onto_dbpedia_results = r.json()


In [3]:
import nltk
from nltk.corpus import wordnet as wn
nltk.download('wordnet')
nltk.download('wordnet_examples')
cat = wn.synsets('cat')[0]
for s in wn.synsets('cat'):
    print(s.name(), s.examples())

cat.n.01 []
guy.n.01 ['a nice guy', "the guy's only doing it for some doll"]
cat.n.03 ['what a cat she is!']
kat.n.01 ['in Yemen kat is used daily by 85% of adults']
cat-o'-nine-tails.n.01 ['British sailors feared the cat']
caterpillar.n.02 []
big_cat.n.01 []
computerized_tomography.n.01 []
cat.v.01 []
vomit.v.01 ['After drinking too much, the students vomited', 'He purged continuously', 'The patient regurgitated the food we gave him last night']


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Marco\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Error loading wordnet_examples: Package 'wordnet_examples'
[nltk_data]     not found in index


In [1]:
import mlflow
mlflow.set_tracking_uri("sqlite:///D:/Projects/OntologyCreation/mlflow.db")
prompt = mlflow.genai.register_prompt(
    name="ontology_extraction",
    template="""You are given different text summaries, each separated by multiple = symbols.
You are building a basic version of an ontology with classes and subclassOf object property.
Identify all classes mentioned across summaries. For each class return all its parents, and not only direct parents.
For example dog sublcassOf Mammal and dog subclassOf Animal are both valid and both should be found if possible in the format Dog: [ Mammal, Animal, etc. ].
Use class names as they appear in text. Include only classes and relations supported by text.
The top level concept here is {{concept}}.
""",
    commit_message="Parametrized the prompt in order to give the top concept",
)

d:\Projects\OntologyCreation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
from OntologyCreation.nlp import term_variants


term_variants("murder")

['murder', 'slaying', 'execution']

In [20]:
synsets = wordnet.synsets("work")
for s in synsets:
    print(s.name(), s.definition())

work.n.01 activity directed toward making or doing something
work.n.02 a product produced or accomplished through the effort or activity or agency of a person or thing
employment.n.02 the occupation for which you are paid
study.n.02 applying the mind to learning and understanding a subject (especially by reading)
work.n.05 (physics) a manifestation of energy; the transfer of energy from one physical system to another expressed as the product of a force and the distance through which it moves a body in the direction of that force
workplace.n.01 a place where work is done
oeuvre.n.01 the total output of a writer or artist (or a substantial part of it)
work.v.01 exert oneself by doing mental or physical work for a purpose or out of necessity
work.v.02 be employed
work.v.03 have an effect or outcome; often the one desired or expected
function.v.01 perform as expected when applied
work.v.05 shape, form, or improve a material
exercise.v.03 give a workout to
make.v.36 proceed along a path
wor

In [7]:
mlflow.set_tracking_uri("sqlite:///D:/Projects/OntologyCreation/mlflow.db")
prompt = mlflow.genai.load_prompt("prompts:/ontology_extraction@latest").format(concept = 'Animal')
prompt

'You are given different text summaries, each separated by multiple = symbols.\nYou are building a basic version of an ontology with classes and subclassOf object property.\nIdentify all classes mentioned across summaries. For each class return all its parents, and not only direct parents.\nFor example dog sublcassOf Mammal and dog subclassOf Animal are both valid and both should be found if they are inside the text.\nUse class names as they appear in text. Include only classes and relations supported by text.\nThe top level concept is Animal.'

In [ ]:
# unique_items = list(set( [x['subclass']['value'] for x in onto_dbpedia_results['results']['bindings']]))
# unique_resource_uris = [x.replace('ontology','resource') for x in unique_items]

# for r_uri in unique_resource_uris:
#     query = """
#     SELECT DISTINCT ?wiki WHERE {
#         <URI> foaf:isPrimaryTopicOf ?wiki .
#     }
#     """.replace("URI", 'Curler')


#     resource_result =requests.get(
#     "https://dbpedia.org/sparql",
#     params={"query": query, "format": "json"}
#     )

#     if resource_result.status_code != 200:
#         continue



#     break


# # query = """
# # SELECT DISTINCT ?wiki WHERE {
# #     <http://dbpedia.org/resource/Archaeology> foaf:isPrimaryTopicOf ?wiki .
# # }
# # """

# # # query="""
# # # SELECT ?property ?value WHERE {
# # #     <http://dbpedia.org/resource/Archaeology> ?property ?value .
# # # }
# # # """



# # results2 = r.json()
# resource_result.json()

{'head': {'link': [], 'vars': ['wiki']},
 'results': {'distinct': False, 'ordered': True, 'bindings': []}}

In [ ]:


headers = {"User-Agent": "MyOntologyProject/1.0"}
data_formatted = {}
data_formatted['classes'] = {}

unique_items = list(set( [x['subclass']['value'] for x in onto_dbpedia_results['results']['bindings']]))

start_uri = "http://dbpedia.org/ontology/CONCEPT".replace("CONCEPT", start_concept)
unique_items += [start_uri]
wiki_found_items = [start_uri]

for item in unique_items:
    short_name = item.split('/')[-1]
    response = requests.get(
    "https://en.wikipedia.org/api/rest_v1/page/summary/{name}".format(name=short_name),
    headers=headers
    )

    if response.status_code !=200:
        continue

    wiki_found_items.append(item)
    summary = response.json()['extract']
    data_formatted['classes'][short_name] = {
        'uri': item,
        'wikipedia_summary': summary
    }
# data_formatted['hierarchy'] = 
data_formatted['hierarchy'] = {x['subclass']['value'].split('/')[-1]: x['parent']['value'].split('/')[-1] 
                               for x in onto_dbpedia_results['results']['bindings']
                               if x['subclass']['value'].split('/')[-1] in wiki_found_items
                               and x['parent']['value'].split('/')[-1] in wiki_found_items
                               }

In [200]:
data_formatted['hierarchy'] = {x['subclass']['value'].split('/')[-1]: x['parent']['value'].split('/')[-1] 
                               for x in results['results']['bindings']
                               if x['subclass']['value'] in wiki_found_items
                               and x['parent']['value']in wiki_found_items
                               }

In [210]:
from OntologyCreation import DATA_PATH
import json

with open(DATA_PATH / f"{start_concept}.json", "w") as f:
    json.dump(data_formatted, f, indent=4)

In [ ]:


r = 

In [105]:
r.json()['extract']

'The cat, also called domestic cat and house cat, is a small carnivorous mammal. It is an obligate carnivore, requiring a predominantly meat-based diet. Its retractable claws are adapted to killing small prey species such as mice and rats. It has a strong, flexible body, quick reflexes, and sharp teeth, and its night vision and sense of smell are well developed. It is a social species, but a solitary hunter and a crepuscular predator.\nCat communication includes meowing, purring, trilling, hissing, growling, grunting, and body language. It can hear sounds too faint or too high in frequency for human ears, such as those made by small mammals. It secretes and perceives pheromones. Cat intelligence is evident in its ability to adapt, learn through observation, and solve problems. \nFemale domestic cats can have kittens from spring to late autumn in temperate zones and throughout the year in equatorial regions, with litter sizes often ranging from two to five kittens.'

In [ ]:
load_dotenv()

True

In [222]:
data['hierarchy']

{'Academic': 'Person',
 'Archeologist': 'Person',
 'Architect': 'Person',
 'Aristocrat': 'Person',
 'Artist': 'Person',
 'Astronaut': 'Person',
 'Athlete': 'Person',
 'Chef': 'Person',
 'Cleric': 'Person',
 'Coach': 'Person',
 'Criminal': 'Person',
 'Economist': 'Person',
 'Egyptologist': 'Person',
 'Engineer': 'Person',
 'Farmer': 'Person',
 'Journalist': 'Person',
 'Judge': 'Person',
 'Lawyer': 'Person',
 'Linguist': 'Person',
 'Man': 'Person',
 'Model': 'Person',
 'Monarch': 'Person',
 'Noble': 'Person',
 'Philosopher': 'Person',
 'Pilot': 'Person',
 'Politician': 'Person',
 'Presenter': 'Person',
 'Producer': 'Person',
 'Psychologist': 'Person',
 'Referee': 'Person',
 'Religious': 'Person',
 'Royalty': 'Person',
 'Scientist': 'Person',
 'Spy': 'Person',
 'Woman': 'Person',
 'Writer': 'Person',
 'Youtuber': 'Person',
 'Person': 'Animal',
 'Amphibian': 'Animal',
 'Arachnid': 'Animal',
 'Bird': 'Animal',
 'Crustacean': 'Animal',
 'Fish': 'Animal',
 'Insect': 'Animal',
 'Mammal': 'Anim

In [223]:
with open(DATA_PATH / f"{'Animal'}.json") as f:
        data = json.load(f)
summaries = [value['wikipedia_summary'] for _, value in data['classes'].items()]
text = '\n'.join(summaries)
print(text)

In Christianity, an archbishop is a bishop of higher rank or office. In most cases, such as the Catholic Church, there are many archbishops who either have jurisdiction over an ecclesiastical province in addition to their own archdiocese, and some who hold non-metropolitan sees or are otherwise granted a titular archbishopric. In others, such as the Lutheran Church of Sweden, the title is borne by the leader of the denomination.
Linguistics is the scientific study of language. The areas of linguistic analysis are syntax, semantics (meaning), morphology, phonetics, phonology, and pragmatics. Subdisciplines such as biolinguistics and psycholinguistics bridge many of these divisions.
Academic or academics may refer to:Academic discipline, a subdivision of knowledge that is taught and researched at the college or university level
Academic institution, an educational institution dedicated to education and research
Academic research, creative and systematic work undertaken to increase the st

In [25]:

class OntologyClasses(BaseModel):
    classes: list[str]
    subclass_of: dict[str, str]  # child -> parent


In [27]:
def onto_to_text(onto) -> str:

    sentences= []

    for cls in onto.classes():
        for axiom in cls.is_a:

            if isinstance(axiom, Restriction):
                if isinstance(axiom.value, Or) :
                    # print(axiom.property.name, list(axiom.value.Classes))
                    sentences.append(f"{cls.name} {axiom.property.name} {list(axiom.value.Classes)}")
                else:
                    # print(axiom.property.name, axiom.value)
                    sentences.append(f"{cls.name} {axiom.property.name} {axiom.value}")

                pass
            else:
                sentences.append(f"{cls.name} is a type of {axiom.name}")

    return '\n'.join(sentences)


In [28]:
def get_ground_truth(onto):
    real_classes = set(c.name for c in onto.classes())
    real_hierarchy = {}
    for cls in onto.classes():
        for axiom in cls.is_a:
            if isinstance(axiom, ThingClass) and axiom.name != "Thing":
                real_hierarchy[cls.name] = axiom.name
    return real_classes, real_hierarchy


In [ ]:
def evaluate(result, real_classes, real_hierarchy):
    extracted_classes = set(result.output.classes)
    extracted_hierarchy = result.output.subclass_of

    class_tp = len(extracted_classes & real_classes)
    class_precision = class_tp / len(extracted_classes) if extracted_classes else 0
    class_recall = class_tp / len(real_classes) if real_classes else 0

    hier_correct = sum(1 for c, p in extracted_hierarchy.items() if real_hierarchy.get(c) == p)
    hier_precision = hier_correct / len(extracted_hierarchy) if extracted_hierarchy else 0
    hier_recall = hier_correct / len(real_hierarchy) if real_hierarchy else 0

    return {
        "class_precision": class_precision,
        "class_recall": class_recall,
        "hierarchy_precision": hier_precision,
        "hierarchy_recall": hier_recall,
    }


In [ ]:


async def run():
    onto = get_ontology('https://raw.githubusercontent.com/owlcs/pizza-ontology/master/pizza.owl').load()
    text = onto_to_text(onto)
    real_classes, real_hierarchy = get_ground_truth(onto)
    agent = Agent(
    "anthropic:claude-haiku-4-5-20251001",
    output_type=OntologyClasses,
    instructions="Extract classes and subclass relationships from the text. Return class names and their parent class."
    )

    model = os.getenv('MODEL')
    with mlflow.start_run():
        mlflow.log_param("ontology", "pizza")
        result = await agent.run(text)

        metrics = evaluate(result, real_classes, real_hierarchy)
        usage = result.usage()

        mlflow.log_metrics(metrics)

        print(metrics)
        print("tokens:", usage.total_tokens)

    return result




In [6]:

agent = Agent(
    os.environ['MODEL'],
    output_type=OntologyClasses,
    instructions="Extract classes and class hierrhcy from text (subclass of relations)"
)

In [ ]:
result = await agent.run(onto_to_text(onto))

In [12]:
result.usage()

RunUsage(input_tokens=5777, output_tokens=2341, details={'input_tokens': 5777, 'output_tokens': 2341, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0}, requests=1)

In [17]:
extracted_classes = set(result.output.classes)
real_classes = set(c.name for c in onto.classes())

print("precision:", len(extracted_classes & real_classes) / len(extracted_classes))
print("recall:", len(extracted_classes & real_classes) / len(real_classes))

precision: 0.99
recall: 1.0


In [21]:
# extracted
extracted_hierarchy = result.output.subclass_of  # your dict child -> parent

# compare
correct = 0
for child, parent in extracted_hierarchy.items():
    if real_hierarchy.get(child) == parent:
        correct += 1

precision = correct / len(extracted_hierarchy) if extracted_hierarchy else 0
recall = correct / len(real_hierarchy) if real_hierarchy else 0
print("hierarchy precision:", precision)
print("hierarchy recall:", recall)

hierarchy precision: 0.8282828282828283
hierarchy recall: 0.9879518072289156
